#Embedding-Based Retrieval with Activeloop and OpenAI

Copyright 2024 Denis Rothman

This second component of the RAG pipeline transforms the prepared data into embeddings and stores the vectors in the vector store.

**Updated to use Deep Lake v4 + LangChain integration.**

# Installing the environment

*First run the following cells. Then run the notebook again cell by cell to explore the code.*

In [ ]:
# Install Deep Lake v4 and LangChain integration
!uv pip install "deeplake>=4.0" langchain-deeplake langchain-openai langchain langchain-community langchain-text-splitters python-dotenv


Load API tokens from `.env` file

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# OpenAI API Key
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# Activeloop Token (required for cloud storage with al://)
os.environ["ACTIVELOOP_TOKEN"] = os.getenv("ACTIVELOOP_TOKEN")

print("API keys loaded.")


grequests.py contains a function to download files from GitHub

In [ ]:
#GitHub grequests.py
#Script to download files from the GitHub repository.

import subprocess

url = "https://raw.githubusercontent.com/Denis2054/RAG-Driven-Generative-AI/main/commons/grequests.py"
output_file = "grequests.py"

curl_command = ["curl", "-o", output_file, url]

try:
    subprocess.run(curl_command, check=True)
    print("Download successful.")
except subprocess.CalledProcessError:
    print("Failed to download the file.")


# Embedding and Storage: populating the vector store

## Downloading and preparing the data

In [ ]:
from grequests import download
source_text = "llm.txt"

directory = "Chapter02"
filename = "llm.txt"
download(directory, filename)


In [ ]:
# Preview the file
with open('llm.txt', 'r', encoding='utf-8') as file:
    lines = file.readlines()
    for line in lines[:20]:
        print(line.strip())


## Setting up LangChain + Deep Lake v4

Deep Lake v4 uses the `langchain-deeplake` integration.
The dataset path uses the `al://` prefix for cloud storage:
- Format: `al://<org_id>/<workspace>/<table_name>`
- Your org_id is the organization name in your Activeloop account.

In [ ]:
# Set your Activeloop organization ID here
# Find it at https://app.activeloop.ai -> click your profile -> Organization
org_id = "mahmoudkamal01099s-organization"  # <-- Replace with your org ID if different

# Dataset path for Deep Lake v4 cloud storage
vector_store_path = f"al://{org_id}/first-workspace/my_table"
print(f"Vector store path: {vector_store_path}")


## Adding data to the vector store

In [ ]:
add_to_vector_store = True


In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_deeplake.vectorstores import DeeplakeVectorStore
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader

# Initialize OpenAI embeddings (Deep Lake v4 uses LangChain-style embeddings)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

if add_to_vector_store:
    # Load and chunk the text file
    loader = TextLoader("llm.txt", encoding="utf-8")
    documents = loader.load()

    text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
    docs = text_splitter.split_documents(documents)

    # Add metadata to each chunk
    for doc in docs:
        doc.metadata["source"] = "llm.txt"

    print(f"Total chunks to embed: {len(docs)}")

    # Create or overwrite the vector store in the cloud
    # overwrite=True clears any previous data at this path
    db = DeeplakeVectorStore.from_documents(
        dataset_path=vector_store_path,
        embedding=embeddings,
        documents=docs,
        overwrite=True,
    )
    print("Vector store created and data uploaded successfully!")
else:
    # Load an existing vector store
    db = DeeplakeVectorStore(
        dataset_path=vector_store_path,
        embedding=embeddings,
    )
    print("Loaded existing vector store.")


# Vector Store information

Summary

In [ ]:
# Show basic info about the vector store
print(f"Vector store path: {vector_store_path}")
print(f"Number of documents stored: {len(db.dataset)}")


In [ ]:
# Inspect the underlying Deep Lake dataset
import deeplake
ds = db.dataset
print("Dataset summary:")
print(ds)


In [ ]:
# Estimate dataset size
try:
    ds_size = ds.size_approx()
    ds_size_mb = ds_size / 1048576
    ds_size_gb = ds_size / 1073741824
    print(f"Dataset size in megabytes: {ds_size_mb:.5f} MB")
    print(f"Dataset size in gigabytes: {ds_size_gb:.5f} GB")
except Exception as e:
    print(f"Size approximation not available: {e}")


# Test: Similarity Search

In [ ]:
# Quick similarity search test to verify everything works
query = "Tell me about space exploration on the Moon and Mars."
results = db.similarity_search(query, k=3)

print(f"Query: {query}\n")
for i, doc in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(f"Source: {doc.metadata.get('source', 'N/A')}")
    print(f"Content: {doc.page_content[:300]}...")
    print()
